In [0]:
CREATE SCHEMA IF NOT EXISTS gold_rag;

In [0]:
DESCRIBE TABLE silver_jobtech.job_postings_silver_dedup;

In [0]:
SELECT
    job_id,
    description,
    description.text AS description_text,
    description.requirements AS description_requirements,
    description.conditions AS description_conditions,
    description.company_information AS description_company_information,
    description.needs AS description_needs
FROM silver_jobtech.job_postings_silver_dedup
WHERE description IS NOT NULL
LIMIT 10;

In [0]:
CREATE OR REPLACE TABLE gold_rag.job_ads_documents AS
SELECT
    job_id,
    job_title,
    employer_name,
    occupation_label,
    occupation_field_label,
    occupation_group_label,
    municipality_code,
    city,

    TO_TIMESTAMP(publication_date) AS publication_date,
    TO_TIMESTAMP(last_publication_date) AS last_publication_date,
    TO_TIMESTAMP(application_deadline) AS application_deadline,
    TO_TIMESTAMP(removed_date) AS removed_date,

    employment_type_label,
    duration_label,
    working_hours_type_label,
    salary_type_label,
    scope_of_work_min,
    scope_of_work_max,
    number_of_vacancies,
    removed,
    webpage_url,
    record_source,

    TRIM(description.text) AS description_text,
    TRIM(description.requirements) AS description_requirements,
    TRIM(description.conditions) AS description_conditions,
    TRIM(description.company_information) AS description_company_information,
    TRIM(description.needs) AS description_needs

FROM silver_jobtech.job_postings_silver_dedup;

## Validation queries

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT job_id) AS distinct_job_ids
FROM gold_rag.job_ads_documents;

In [0]:
SELECT
    job_id,
    COUNT(*) AS occurrences
FROM gold_rag.job_ads_documents
GROUP BY job_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;

In [0]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN description_text IS NOT NULL AND description_text <> '' THEN 1 ELSE 0 END) AS has_description_text,
    SUM(CASE WHEN description_requirements IS NOT NULL AND description_requirements <> '' THEN 1 ELSE 0 END) AS has_requirements,
    SUM(CASE WHEN description_conditions IS NOT NULL AND description_conditions <> '' THEN 1 ELSE 0 END) AS has_conditions,
    SUM(CASE WHEN description_company_information IS NOT NULL AND description_company_information <> '' THEN 1 ELSE 0 END) AS has_company_information,
    SUM(CASE WHEN description_needs IS NOT NULL AND description_needs <> '' THEN 1 ELSE 0 END) AS has_needs
FROM gold_rag.job_ads_documents;

In [0]:
SELECT
    job_id,
    job_title,
    employer_name,
    occupation_label,
    city,
    description_text,
    description_requirements,
    description_conditions
FROM gold_rag.job_ads_documents
LIMIT 10;

- `job_id` uniqueness is clean
- `description_text` has very strong coverage
- `description_conditions` is also quite valuable
- `description_company_information` exists but is much sparser
- `description_requirements` and `description_needs` are effectively unused in this dataset

next move is to build a retrieval-ready document_text that prioritizes:

- core metadata
- main ad body
- conditions
- company info only when present

We would not force in `requirements` or `needs` right now since they are zero anyway.

## create a retrieval-friendly text blob

In [0]:
CREATE OR REPLACE TABLE gold_rag.job_ads_documents_enriched AS
SELECT
    job_id,
    job_title,
    employer_name,
    occupation_label,
    occupation_field_label,
    occupation_group_label,
    municipality_code,
    city,
    publication_date,
    last_publication_date,
    application_deadline,
    removed_date,
    employment_type_label,
    duration_label,
    working_hours_type_label,
    salary_type_label,
    scope_of_work_min,
    scope_of_work_max,
    number_of_vacancies,
    removed,
    webpage_url,
    record_source,
    description_text,
    description_requirements,
    description_conditions,
    description_company_information,
    description_needs,

    CONCAT_WS(
        '\n\n',

        CONCAT('Job title: ', COALESCE(job_title, '')),
        CONCAT('Employer: ', COALESCE(employer_name, '')),
        CONCAT('Occupation: ', COALESCE(occupation_label, '')),
        CONCAT('Occupation field: ', COALESCE(occupation_field_label, '')),
        CONCAT('Occupation group: ', COALESCE(occupation_group_label, '')),
        CONCAT('Location: ', COALESCE(city, '')),
        CONCAT('Municipality code: ', COALESCE(municipality_code, '')),
        CONCAT('Employment type: ', COALESCE(employment_type_label, '')),
        CONCAT('Duration: ', COALESCE(duration_label, '')),
        CONCAT('Working hours: ', COALESCE(working_hours_type_label, '')),
        CONCAT('Salary type: ', COALESCE(salary_type_label, '')),
        CONCAT('Vacancies: ', COALESCE(CAST(number_of_vacancies AS STRING), '')),
        CONCAT('Publication date: ', COALESCE(CAST(publication_date AS STRING), '')),
        CONCAT('Last publication date: ', COALESCE(CAST(last_publication_date AS STRING), '')),
        CONCAT('Application deadline: ', COALESCE(CAST(application_deadline AS STRING), '')),

        CASE
            WHEN description_text IS NOT NULL AND TRIM(description_text) <> ''
            THEN CONCAT('Job description: ', description_text)
        END,

        CASE
            WHEN description_conditions IS NOT NULL AND TRIM(description_conditions) <> ''
            THEN CONCAT('Conditions: ', description_conditions)
        END,

        CASE
            WHEN description_company_information IS NOT NULL AND TRIM(description_company_information) <> ''
            THEN CONCAT('Company information: ', description_company_information)
        END
    ) AS document_text

FROM gold_rag.job_ads_documents;

## Validation after creation

In [0]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN document_text IS NOT NULL AND TRIM(document_text) <> '' THEN 1 ELSE 0 END) AS has_document_text
FROM gold_rag.job_ads_documents_enriched;

In [0]:
SELECT
    job_id,
    job_title,
    employer_name,
    city,
    document_text
FROM gold_rag.job_ads_documents_enriched
LIMIT 5;

In [0]:
SELECT
    MIN(LENGTH(document_text)) AS min_length,
    MAX(LENGTH(document_text)) AS max_length,
    AVG(LENGTH(document_text)) AS avg_length
FROM gold_rag.job_ads_documents_enriched;

In [0]:
SELECT
    CASE
        WHEN LENGTH(document_text) < 500 THEN '<500'
        WHEN LENGTH(document_text) < 1000 THEN '500-999'
        WHEN LENGTH(document_text) < 2000 THEN '1000-1999'
        WHEN LENGTH(document_text) < 4000 THEN '2000-3999'
        ELSE '4000+'
    END AS length_bucket,
    COUNT(*) AS row_count
FROM gold_rag.job_ads_documents_enriched
GROUP BY
    CASE
        WHEN LENGTH(document_text) < 500 THEN '<500'
        WHEN LENGTH(document_text) < 1000 THEN '500-999'
        WHEN LENGTH(document_text) < 2000 THEN '1000-1999'
        WHEN LENGTH(document_text) < 4000 THEN '2000-3999'
        ELSE '4000+'
    END
ORDER BY length_bucket;

## Results
- document_text coverage is complete
- the text quality looks good for retrieval
- corpus is quite long on average
- most ads are in the 2000–3999 character range
- a very large share are 4000+, so chunking will matter for embeddings

## Decision
We build a chunk table.
We do not stop at document-level retrieval since our next step is embeddings + semantic search.

Distribution shows:

- average ~2952 chars
- max 7773 chars
- over 1.36 million ads are 4000+ chars

That means document-level embeddings are possible, but they are not ideal. 
- Long ads will dilute signal. 
- A query about one specific skill, condition, or requirement may get buried inside a long full-document embedding.

# Gold RAG – Document Layer (Handover)

## Purpose

This layer prepares job advertisements for Retrieval-Augmented Generation (RAG).

It is designed to:
- preserve rich textual content from job ads
- include relevant metadata for filtering and context
- provide one document per job ad (no duplication)
- serve as a clean input for downstream chunking and embeddings

---

## Source

**Input table:**
silver_jobtech.job_postings_silver_dedup

**Reason:**
- deduplicated on `job_id`
- retains full `description` struct
- contains both structured metadata and raw text

---

## Gold RAG Tables

### 1. Base Document Table

**Table:**
gold_rag.job_ads_documents

**Description:**
- one row per job ad (`job_id`)
- includes structured metadata
- flattens description struct into usable text fields

**Key columns:**
- job_id
- job_title
- employer_name
- occupation_label
- occupation_field_label
- occupation_group_label
- municipality_code
- city
- publication_date
- last_publication_date
- application_deadline
- employment_type_label
- duration_label
- working_hours_type_label
- salary_type_label
- number_of_vacancies
- record_source

**Flattened description fields:**
- description_text (main content)
- description_conditions
- description_company_information
- description_requirements (currently empty)
- description_needs (currently empty)

---

### 2. Retrieval-Ready Document Table

**Table:**
gold_rag.job_ads_documents_enriched

**Adds:**
document_text

This is a structured text field used for retrieval and future chunking.

---

## document_text Structure

Each document is formatted as:

Job title: ...

Employer: ...

Occupation: ...

Occupation field: ...

Occupation group: ...

Location: ...

Municipality code: ...

Employment type: ...

Duration: ...

Working hours: ...

Salary type: ...

Vacancies: ...

Publication date: ...

Last publication date: ...

Application deadline: ...


Job description:

...

Conditions:

...

Company information:

...

---

## Data Characteristics & Validation

- Total rows: ~6.75 million
- Unique job_id: 100% (no duplicates)
- document_text coverage: 100%
- Average document length: ~2950 characters
- Max length: ~7700 characters

**Length distribution:**
- majority: 2000–4000 characters
- large portion: 4000+ characters

**Text field coverage:**
- description_text → ~99% populated
- description_conditions → ~73% populated
- description_company_information → low coverage
- description_requirements → 0%
- description_needs → 0%

---

## Important Design Decisions

### 1. Separate RAG layer from analytics

This layer does NOT reuse star schema tables.

Reason:
- analytics requires normalized structures (facts/dimensions)
- RAG requires denormalized, text-rich documents

---

### 2. No chunking implemented yet

This layer is intentionally document-level only.

Reason:
- chunking depends on embedding strategy and model choice
- handled downstream

---

### 3. Metadata included in document_text

Metadata is embedded directly in the text to:
- improve semantic retrieval
- provide better context for LLM responses
- support zero-filter retrieval scenarios

---

## Recommended Next Steps (for continuation)

### 1. Chunking

Split document_text into smaller segments.

Suggested starting configuration:
- chunk size: 1000–1200 characters
- overlap: ~200 characters

---

### 2. Chunk Table

Create:

gold_rag.job_ads_chunks

Suggested structure:
- job_id
- chunk_id
- chunk_index
- chunk_text
- metadata columns (location, occupation, etc.)

---

### 3. Embeddings

- generate embeddings per chunk (not per full document)
- store vectors alongside metadata

---

### 4. Retrieval Strategy

- semantic search over chunks
- optional metadata filtering (e.g. location, occupation)
- ranking based on similarity

---

## Summary

This layer provides:

- clean, deduplicated job ads
- structured + unstructured data combined
- retrieval-ready documents
- strong foundation for chunking and embeddings
